# Notebook 1 · Who are the SBLCs? (Question 1)

**Question:** The data does not label Small Business Lending Companies (SBLCs). Identify them, show how you decided, and say how confident you are.

**What an SBLC is:** a *non-depository* lender (no deposits, no FDIC or NCUA id) licensed by SBA to make 7(a) loans.

**Approach: a four-step funnel**
1. **Profile every lender** (size, geography, product mix, loan size).
2. **Gate:** remove banks, credit unions and government entities. An SBLC cannot be one of these.
3. **Score** the remaining lenders on how much they look like an SBLC, using only rules visible in the data.
4. **Cross-check** against public sources and assign a **confidence tier**.

**Principle:** we never use loan *outcomes* (defaults) to identify SBLCs. Identification must not depend on the credit results we analyse later.

**Input:** `data/processed/loans_clean.parquet` (Notebook 0)
**Outputs:** `assignment1/outputs/sblc_list.csv`, `assignment1/outputs/sblc_candidates_scored.csv`

In [17]:
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

## Step 0 · Load the clean data
Duplicates, cancelled and undisbursed loans were already removed in Notebook 0.

In [18]:
df = pd.read_parquet("../../data/processed/loans_clean.parquet")

print("Loans:", len(df), "| distinct lender names:", df["bankname"].nunique())

Loans: 478650 | distinct lender names: 2758


## Step 1 · Profile every lender

One row per lender name. The columns capture the things that should separate an SBLC from a bank or a community lender:
- **FDIC / NCUA id:** regulated depositories have one.
- **Number of borrower states:** SBLCs lend nationally, while community lenders are regional.
- **Community Advantage share:** a pilot program for mission-driven lenders, which are not SBLCs.
- **PLP share and median loan size:** commercial lenders use the Preferred Lenders Program and larger loans.

In [19]:
g = df.groupby("bankname")

lenders = pd.DataFrame({
    "loans": g.size(),
    "gross_usd_m": g["grossapproval"].sum() / 1e6,
    "median_loan": g["grossapproval"].median(),
    "n_borrower_states": g["borrstate"].nunique(),
    "has_fdic": g["bankfdicnumber"].apply(lambda s: s.notna().any()),
    "has_ncua": g["bankncuanumber"].apply(lambda s: s.notna().any()),
    "ca_share": g["processingmethod"].apply(lambda s: (s == "Community Advantage Initiative").mean()),
    "plp_share": g["processingmethod"].apply(
        lambda s: s.isin(["Preferred Lenders Program", "Preferred Lenders with EWCP"]).mean()),
    "express_share": g["processingmethod"].apply(lambda s: s.str.contains("Express", na=False).mean()),
    "first_fy": g["approvalfy"].min(),
    "last_fy": g["approvalfy"].max(),
    "n_location_ids": g["locationid"].nunique(),
}).reset_index()

lenders.sort_values("loans", ascending=False).head(8)

,bankname,loans,gross_usd_m,median_loan,n_borrower_states,has_fdic,has_ncua,ca_share,plp_share,express_share,first_fy,last_fy,n_location_ids
2687,Wells Fargo Bank National Association,39713,13069.0651,50000.0,52,True,False,0.0,0.362325,0.565407,2010,2019,1
2408,The Huntington National Bank,33811,7502.0240,50000.0,48,True,False,0.0,0.187543,0.751738,2010,2019,1
1362,"JPMorgan Chase Bank, National Association",32078,5706.7760,75000.0,45,True,False,0.0,0.097232,0.880011,2010,2019,1
2544,"U.S. Bank, National Association",20223,3670.0654,30000.0,44,True,False,0.0,0.148346,0.803738,2010,2019,1
2215,"TD Bank, National Association",13822,1923.2821,50000.0,27,True,False,0.0,0.070323,0.919259,2010,2019,1
1772,"PNC Bank, National Association",13713,3182.1359,50000.0,49,True,False,0.0,0.189455,0.796543,2010,2019,1
1517,Manufacturers and Traders Trust Company,12849,1764.1868,50000.0,25,True,False,0.0,0.117130,0.788466,2010,2019,1
2533,Truist Bank,6971,2822.3912,190000.0,31,True,False,0.0,0.353751,0.552288,2010,2019,1


## Step 2 · The gate: remove depositories and government entities
SBLCs are by definition non-depository, so any lender with an FDIC or NCUA id is out. We also remove the FDIC, NCUA and SBA servicing entities, which appear as "lenders" when they take over loans from failed institutions.

**Check the gate for leaks:** some banks and credit unions are missing an id in the file. We list lenders that survive the gate but have bank-like names, to see how big this problem is.

In [20]:
govt = lenders["bankname"].str.contains(r"Federal Deposit|National Credit Union|\bSBA CLSC",
                                        case=False, regex=True)
lenders["depository"] = lenders["has_fdic"] | lenders["has_ncua"]
lenders["excluded"] = lenders["depository"] | govt

pool = lenders[~lenders["excluded"]].copy()      # candidates that survive the gate

funnel = pd.DataFrame({
    "step": ["All lenders", "Have FDIC/NCUA id (banks, credit unions)",
             "Government entities (FDIC, NCUA, SBA)", "Non-depository pool (candidates)"],
    "lenders": [len(lenders), lenders["depository"].sum(),
                (govt & ~lenders["depository"]).sum(), len(pool)],
    "loans": [lenders["loans"].sum(), lenders.loc[lenders["depository"], "loans"].sum(),
              lenders.loc[govt & ~lenders["depository"], "loans"].sum(), pool["loans"].sum()],
})
funnel["loans_%"] = (funnel["loans"] / len(df) * 100).round(1)
display(funnel)

# Gate leak check: bank-like names that still made it into the pool
leaks = pool[pool["bankname"].str.contains(r"\bbank\b|trust company|savings|credit union|\bFCU\b|\bCU\b",
                                            case=False, regex=True)]
print(f"Bank-like names that survived the gate: {len(leaks)}")
leaks[["bankname", "loans", "n_borrower_states"]].sort_values("loans", ascending=False).head(8)

,step,lenders,loans,loans_%
0,All lenders,2758,478649,100.0
1,"Have FDIC/NCUA id (banks, credit unions)",2559,459959,96.1
2,"Government entities (FDIC, NCUA, SBA)",3,141,0.0
3,Non-depository pool (candidates),196,18549,3.9


Bank-like names that survived the gate: 43


,bankname,loans,n_borrower_states
763,Economic Development Bank for Puerto Rico,267,1
2192,Summit Community Bank,31,1
2296,The Citizens National Bank of Paris,20,2
1459,"Liberty Bank, FSB",17,2
2649,Voyager Bank,16,2
2567,Union Settlement FCU,16,2
321,Beehive CU,12,1
1967,RiverWood Bank,12,1


## Step 3 · Score each candidate
Each rule adds or removes points. Every rule uses only information in the file, and every point is visible in the table below, so the decision is transparent.

| Rule | Points | Reasoning |
|---|---|---|
| Name contains "SBLC" | +3 | Direct hint |
| Name has small-business lending / loan / finance wording | +2 | Weaker hint |
| Name looks like a bank | -1 | Probably a depository missing its id |
| Name looks like a CDC, development corporation, BIDCO, farm credit, fund, etc. | -2 | A different kind of lender |
| Lends in 10+ states (or 5-9) | +2 (or +1) | SBLCs are national |
| 50%+ of loans under Community Advantage | -2 (else +1) | Mission-lender pilot, not SBLC |
| Mostly PLP loans, or median loan of $150k+ | +1 | Commercial lender profile |
| 100+ loans | +1 | Enough evidence to judge |

In [21]:
NAME_SBLC = r"SBLC"
NAME_LEND = r"small business (?:lend|loan|financ)|SBA lending|business loan"
NAME_NEG = (r"develop|economic|BIDCO|farm|AgSouth|AgChoice|AgCredit|Ag Credit|ACA\b|FLCA|certified|community|"
            r"initiative|foundation|fund\b|housing|enterprise|entrepreneur|trade finance|CDC\b|CDFI|"
            r"\bcorp(?:oration)? of\b|credit union|\bFCU\b|\bCU\b")
NAME_BANK = r"\bbank\b|trust company|savings|national association"

name = pool["bankname"]
pool["pts_name"] = np.where(name.str.contains(NAME_SBLC, case=False, regex=True), 3,
                    np.where(name.str.contains(NAME_LEND, case=False, regex=True), 2, 0))
pool["pts_neg_name"] = np.where(name.str.contains(NAME_NEG, case=False, regex=True), -2, 0)
pool["pts_bank_name"] = np.where(name.str.contains(NAME_BANK, case=False, regex=True), -1, 0)
pool["pts_footprint"] = np.select([pool["n_borrower_states"] >= 10, pool["n_borrower_states"] >= 5], [2, 1], default=0)
pool["pts_ca"] = np.where(pool["ca_share"] >= 0.5, -2, 1)
pool["pts_commercial"] = np.where((pool["plp_share"] >= 0.5) | (pool["median_loan"] >= 150_000), 1, 0)
pool["pts_scale"] = np.where(pool["loans"] >= 100, 1, 0)

rules = [c for c in pool.columns if c.startswith("pts_")]
pool["score"] = pool[rules].sum(axis=1)

# Show the points behind the top scores
pool.sort_values(["score", "loans"], ascending=False).head(8).set_index("bankname")[rules + ["score"]]

,pts_name,pts_neg_name,pts_bank_name,pts_footprint,pts_ca,pts_commercial,pts_scale,score
bankname,,,,,,,,
"First Western SBLC, Inc",3,0,0,2,1,1,1,8
"Newtek Small Business Finance, Inc.",2,0,0,2,1,1,1,7
"Hana Small Business Lending, Inc.",2,0,0,2,1,1,1,7
"CRF Small Business Loan Company, LLC",2,0,0,2,1,1,1,7
"Harvest Small Business Finance, LLC",2,0,0,2,1,1,1,7
"Centerstone SBA Lending, Inc.",2,0,0,2,1,1,1,7
"CapitalSpring SBLC, LLC",3,0,0,1,1,1,0,6
"Readycap Lending, LLC",0,0,0,2,1,1,1,5


## Step 4 · Cross-check with public sources
The data alone cannot prove a lender is licensed. We add one more piece of evidence: is the lender named as an SBA-licensed SBLC in a public source?

**Sources used** (found by web search):
- CU Times, 2003: SBA's roster of the 14 SBLCs then licensed (includes Newtek, The Loan Source, Business Loan Center).
- Coleman Report, 2015: ReadyCap and CenterStone SBLC licenses.
- Funderintel: roster of long-time SBLC license holders (ReadyCap, Harvest, Lendistry, VelocitySBA, Exos, CenterStone, First Western SBLC, CRF, Grow America Fund).

**Limitations:** these are secondary sources, so the list should be checked against SBA's official SBLC roster before presenting. A lender can hold more than one license, and the names in this file are brand names.

In [22]:
EXTERNAL = {
    "newtek small business":    "CU Times 2003 / Coleman 2015",
    "loan source incorporated": "CU Times 2003 / Coleman 2015",
    "first western sblc":       "CU Times 2003 / Funderintel 2026",
    "grow america fund":        "CU Times 2003 / Coleman 2015 / Funderintel 2026",
    "readycap":                 "Coleman 2015 / Funderintel 2026",
    "crf small business":       "Coleman 2015 / Funderintel 2026",
    "hana small business":      "Coleman 2015",
    "capitalspring sblc":       "Coleman 2015 (as 'Capital Spring')",
    "harvest small business":   "Funderintel 2026",
    "lendistry":                "Funderintel 2026",
    "velocitysba":              "Funderintel 2026",
    "centerstone sba":          "Funderintel 2026",
    "exos small business":      "Funderintel 2026",
    "business loan center":     "CU Times 2003",
}

lower = pool["bankname"].str.lower()
pool["external_source"] = lower.map(lambda s: next((v for k, v in EXTERNAL.items() if k in s), ""))
pool["external_match"] = pool["external_source"] != ""

# Names from the public sources that do NOT appear in this data
print("Not found in data:", [k for k in EXTERNAL if not lower.str.contains(k, regex=False).any()])

def assign_tier(r):
    if r.external_match and r.score >= 3: return "1-Confirmed"   # data AND public source agree
    if r.score >= 5:                      return "2-Likely"      # data only, strong
    if r.score >= 3:                      return "3-Possible"    # data only, weak
    return "4-Not SBLC"

pool["tier"] = pool.apply(assign_tier, axis=1)

Not found in data: ['exos small business', 'business loan center']


## Results · tiers and top candidates
- **1-Confirmed:** data score of 3 or more **and** named in a public source.
- **2-Likely:** score of 5 or more, no outside confirmation.
- **3-Possible:** score of 3-4, no outside confirmation. Needs manual review.
- **4-Not SBLC:** everything else.

In [23]:
tier_summary = (pool.groupby("tier")
                .agg(lenders=("bankname", "size"), loans=("loans", "sum"), gross_usd_m=("gross_usd_m", "sum"))
                .round(0))
display(tier_summary)

cols = ["bankname", "loans", "n_borrower_states", "ca_share", "plp_share",
        "median_loan", "score", "external_match", "tier"]
cands = pool[pool["score"] >= 3].sort_values(["score", "loans"], ascending=False)
cands[cols].round(2)

,lenders,loans,gross_usd_m
tier,,,
1-Confirmed,12,11465,5978.0
3-Possible,2,143,74.0
4-Not SBLC,182,6941,1890.0


,bankname,loans,n_borrower_states,ca_share,plp_share,median_loan,score,external_match,tier
1079,"First Western SBLC, Inc",295,34,0.00,0.88,1364000.0,8,True,1-Confirmed
1671,"Newtek Small Business Finance, Inc.",2874,52,0.00,0.91,405500.0,7,True,1-Confirmed
1237,"Hana Small Business Lending, Inc.",671,20,0.00,0.86,528000.0,7,True,1-Confirmed
388,"CRF Small Business Loan Company, LLC",494,43,0.02,0.67,300000.0,7,True,1-Confirmed
1245,"Harvest Small Business Finance, LLC",478,34,0.00,0.88,927300.0,7,True,1-Confirmed
450,"Centerstone SBA Lending, Inc.",205,14,0.00,0.80,850000.0,7,True,1-Confirmed
423,"CapitalSpring SBLC, LLC",15,8,0.00,0.40,483000.0,6,True,1-Confirmed
1930,"Readycap Lending, LLC",975,47,0.00,0.82,662000.0,5,True,1-Confirmed
2621,"VelocitySBA, LLC",5008,51,0.00,0.01,10000.0,4,True,1-Confirmed
1455,"Lendistry SBLC, LLC",140,5,0.99,0.01,219000.0,4,True,1-Confirmed


## The picture
Each bubble is a non-depository lender (size = number of loans, color = Community Advantage share). SBLCs should sit **to the right** (many states) and be **dark** (not Community Advantage). Diamonds with labels mark lenders with score 3 or more. Hover over any bubble for details.

In [24]:
plot_df = pool.copy()
is_cand = plot_df["score"] >= 3
plot_df["candidate"] = np.where(is_cand, "Score 3+ (candidate)", "Other non-depository")
plot_df["label"] = np.where(is_cand, plot_df["bankname"].str.split(",").str[0].str.slice(0, 24), "")

fig = px.scatter(
    plot_df,
    x="n_borrower_states", y="median_loan",
    size="loans", color="ca_share",
    symbol="candidate", text="label",
    log_y=True, hover_name="bankname",
    hover_data={"loans": ":,", "score": True, "plp_share": ":.0%", "ca_share": ":.0%",
                "candidate": False, "label": False},
    color_continuous_scale="Viridis", size_max=40,
    labels={"n_borrower_states": "Number of borrower states",
            "median_loan": "Median loan size ($, log scale)",
            "ca_share": "Community Advantage share"},
    title="Non-depository lenders: footprint vs loan size",
)
fig.update_traces(textposition="top center", textfont_size=9)
fig.update_layout(height=550, template="plotly_white")
fig.show()

## How sensitive is the list to our rules?
We remove one rule at a time and see which lenders enter or leave the candidate list (score 3 or more).
- **n_added** large -> that rule protects us from false positives.
- **n_removed** large -> the list depends on that rule to keep true SBLCs.
We also vary the score cutoff.

In [25]:
base = set(pool.loc[pool["score"] >= 3, "bankname"])

rows = []
for r in rules:
    alt = set(pool.loc[(pool["score"] - pool[r]) >= 3, "bankname"])
    rows.append({"rule removed": r, "n_added": len(alt - base), "n_removed": len(base - alt),
                 "removed": ", ".join(sorted(base - alt))[:80]})
display(pd.DataFrame(rows))

print("Candidates by score cutoff:")
print({cut: int((pool["score"] >= cut).sum()) for cut in [2, 3, 4, 5, 6]})

,rule removed,n_added,n_removed,removed
0,pts_name,0,1,"Lendistry SBLC, LLC"
1,pts_neg_name,8,0,
2,pts_bank_name,1,0,
3,pts_footprint,0,4,"Grow America Fund, Incorporated, Loan Source Incorporated, Port 51..."
4,pts_ca,3,4,"ACC Capital, Grow America Fund, Incorporated, Loan Source Incorpor..."
5,pts_commercial,0,4,"ACC Capital, Grow America Fund, Incorporated, Loan Source Incorpor..."
6,pts_scale,0,2,"ACC Capital, Grow America Fund, Incorporated"


Candidates by score cutoff:
{2: 24, 3: 14, 4: 10, 5: 8, 6: 7}


## Final list and save
The working SBLC list is **tiers 1 and 2**. Tier 3 lenders are kept in the scored file for review but not treated as SBLCs in later notebooks.

In [26]:
final = pool[pool["tier"].isin(["1-Confirmed", "2-Likely"])].sort_values("loans", ascending=False)

out = Path("../../assignment1/outputs"); out.mkdir(parents=True, exist_ok=True)
pool.sort_values(["score", "loans"], ascending=False).to_csv(out / "sblc_candidates_scored.csv", index=False)
final[["bankname", "tier", "score", "external_source", "loans", "gross_usd_m"]].to_csv(out / "sblc_list.csv", index=False)

print(f"SBLCs identified: {len(final)}")
print(f"Their loans: {final['loans'].sum():,} ({final['loans'].sum() / len(df):.1%} of all clean loans), "
      f"${final['gross_usd_m'].sum() / 1e3:.1f}bn approved")
final[["bankname", "tier", "score", "external_source", "loans", "gross_usd_m"]].round(1)

SBLCs identified: 12
Their loans: 11,465 (2.4% of all clean loans), $6.0bn approved


,bankname,tier,score,external_source,loans,gross_usd_m
2621,"VelocitySBA, LLC",1-Confirmed,4,Funderintel 2026,5008,130.9
1671,"Newtek Small Business Finance, Inc.",1-Confirmed,7,CU Times 2003 / Coleman 2015,2874,2445.3
1930,"Readycap Lending, LLC",1-Confirmed,5,Coleman 2015 / Funderintel 2026,975,963.5
1237,"Hana Small Business Lending, Inc.",1-Confirmed,7,Coleman 2015,671,624.4
388,"CRF Small Business Loan Company, LLC",1-Confirmed,7,Coleman 2015 / Funderintel 2026,494,247.4
1245,"Harvest Small Business Finance, LLC",1-Confirmed,7,Funderintel 2026,478,607.8
1079,"First Western SBLC, Inc",1-Confirmed,8,CU Times 2003 / Funderintel 2026,295,462.6
1221,"Grow America Fund, Incorporated",1-Confirmed,3,CU Times 2003 / Coleman 2015 / Funderintel 2026,268,154.2
450,"Centerstone SBA Lending, Inc.",1-Confirmed,7,Funderintel 2026,205,253.6
1455,"Lendistry SBLC, LLC",1-Confirmed,4,Funderintel 2026,140,28.7


## Summary and confidence

**Answer:** 12 SBLCs, all **Confirmed** (the data profile and at least one public roster agree). Together they made about 11,500 loans (about 2.4% of loans) and about $6bn of approvals.

**How we decided:** from 2,758 lenders, 2,559 are depositories (FDIC or NCUA id), leaving a pool of 196 non-depository lenders (about 3.9% of loans). We scored the pool on name, national footprint, product mix and size, then cross-checked with public rosters: CU Times 2003, Coleman Report 2015 and Funderintel 2026. Tier counts: 12 Confirmed, 0 Likely, 2 Possible, 182 Not SBLC.

**Confidence**
- **High (all 12):** two independent kinds of evidence agree. Each lender passes the gate, scores 3 or more, and appears on at least one published roster.
- **Borderline (ACC Capital, Port 51 Lending):** score 3 with no outside match under those names. Excluded. Both are on the 2026 active roster but were probably licensed after FY2019, so they should be checked manually.

**What the sensitivity test shows**
- The data alone would have missed several true SBLCs. Velocity, Grow America and Loan Source depend on the national-footprint rule, and Lendistry depends on its "SBLC" name, because it does 99% of its lending under Community Advantage, which normally points to a mission lender.
- The name penalty and the Community Advantage rule protect us from false positives. Check the exact counts in the sensitivity table above before quoting them.

**Limits to keep in mind**
- The rosters are trade-press and industry lists, not the SBA's own SBLC roster. Verify against the SBA's official list before presenting.
- Two roster names (Exos, Business Loan Center) do not appear under those names in the data. One licence can sit behind several brand names.
- Velocity is on the 2026 roster but not the 2015 one, and holds 4,643 FY2010-12 loans. The bank-name field shows the current holder, so part of that early book may have been originated by others.
- Fountainhead SBF and Fund-Ex Solutions (inactive licences) made 3 and 4 loans in FY2019. They are not on the list and should be reviewed.
- The gate leaks a little: some banks and credit unions lack an id in the file, so a few bank-like names survive into the pool. The name rules push them down.
- Newtek converted to a bank after our data window, so it is treated as an SBLC for FY2010-2019.